# IGN (InteractionGraphNet) 虚拟筛选教程

## 概述

InteractionGraphNet (IGN) 是一种基于**交互图神经网络**的蛋白-配体结合亲和力预测方法。

> **论文**: *InteractionGraphNet: A Novel and Efficient Deep Graph Representation Learning Framework for Accurate Protein-Ligand Interaction Predictions* (Jiang et al., 2021)  
> **核心创新**: 将蛋白质口袋原子和配体原子分别作为两组节点，以空间距离阈值筛选蛋白-配体原子对构建**交互图的边**，通过**边级别 (edge-level)** 的消息传递与聚合来预测结合亲和力。

### 与传统 GNN 的关键区别

传统 GNN 通常在**节点**上做消息传递和聚合，而 IGN 的核心思想是：蛋白-配体的结合亲和力主要由接触界面（即**边**）决定，因此模型直接在**边上**聚合信息，而非节点级别。

### 学习目标

通过本教程，你将掌握：

1. 如何从 PDB/MOL2 文件中提取蛋白-配体原子特征
2. 如何基于空间距离阈值构建交互图
3. IGN 的边级别消息传递与注意力聚合机制
4. 结合亲和力预测的训练与测试流程

---

In [ ]:
# ====== 项目根目录定位与路径设置 ======

import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from rdkit import RDLogger
from scipy.spatial import distance_matrix
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

%matplotlib inline

# 抑制 RDKit 的冗余警告信息
RDLogger.logger().setLevel(RDLogger.ERROR)
warnings.filterwarnings('ignore')


def find_project_root(marker='demo_data'):
    """向上查找包含 demo_data/ 的项目根目录"""
    here = Path('.').resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f'无法找到包含 {marker}/ 的项目根目录')


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'demo_data'
CORESET_FILE = DATA_DIR / 'CoreSet.dat'
COMPLEX_DIR = DATA_DIR / 'coreset'

print(f'项目根目录: {PROJECT_ROOT}')
print(f'数据目录:   {DATA_DIR}')
print(f'标签文件:   {CORESET_FILE}')
print(f'复合物目录: {COMPLEX_DIR}')
print()

# ====== 环境检查 ======
checks = {
    '依赖库': ['版本'],
    'Python': [sys.version.split()[0]],
    'PyTorch': [torch.__version__ + f' (CUDA: {torch.cuda.is_available()})'],
    'RDKit': [Chem.rdBase.rdkitVersion],
    'NumPy': [np.__version__],
    'Pandas': [pd.__version__],
}
display(pd.DataFrame(checks).set_index('依赖库').T)

## 1. 超参数设置

IGN 的关键超参数包括：

| 参数 | 含义 | 说明 |
|------|------|------|
| `DISTANCE_CUTOFF` | 交互图边的距离阈值 | 只有蛋白-配体原子对距离 < 该值时，才在两者间建立边。原始论文中常用 5.0 A |
| `ATOM_FEAT_DIM` | 原子特征维度 | 简化为 10 维：8 种常见元素 one-hot + 1 种 "other" + 1 芳香性标记 |
| `HIDDEN_DIM` | 隐层维度 | 节点嵌入和边 MLP 的隐藏层维度 |
| `N_EPOCHS` | 训练轮数 | 在 285 个样本的小数据集上训练 |
| `BATCH_SIZE` | 批大小 | 因为每个复合物的图大小不同（变长图），设为 1 逐样本处理 |

In [ ]:
# ====== 超参数定义 ======

DISTANCE_CUTOFF = 5.0       # 交互图边的距离阈值 (Angstrom)
ATOM_FEAT_DIM = 10          # 简化原子特征维度
HIDDEN_DIM = 128            # 隐层维度
N_EPOCHS = 200              # 训练轮数
LR = 1e-3                   # 学习率
BATCH_SIZE = 1              # 变长图，逐样本处理
SEED = 42                   # 随机种子（保证可复现）
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 固定随机种子
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"设备:        {DEVICE}")
print(f"距离阈值:    {DISTANCE_CUTOFF} A")
print(f"原子特征维度: {ATOM_FEAT_DIM}")
print(f"隐层维度:    {HIDDEN_DIM}")
print(f"训练轮数:    {N_EPOCHS}")
print(f"学习率:      {LR}")
print(f"批大小:      {BATCH_SIZE}")

## 2. 数据加载与特征提取

### 数据来源

本教程使用 **PDBbind CASF-2016 核心集** (core set)，包含 285 个蛋白-配体复合物，每个复合物有：
- `{pdbid}_protein.pdb` — 蛋白质全链结构
- `{pdbid}_pocket.pdb` — 蛋白质结合口袋（配体周围残基）
- `{pdbid}_ligand.mol2 / .sdf` — 配体分子结构
- `CoreSet.dat` 中记录的实验结合亲和力 logKa

### 原子特征设计

每个原子编码为 10 维特征向量：
- **前 9 维**: 元素类型 one-hot 编码 (C, N, O, S, F, P, Cl, Br, other)
- **第 10 维**: 是否为芳香性原子 (0 或 1)

### 交互图构建

对于每个复合物：
1. 提取蛋白口袋和配体的 3D 坐标
2. 计算配体原子到蛋白原子的距离矩阵
3. 距离 < `DISTANCE_CUTOFF` 的原子对形成交互边
4. 边特征为对应的欧氏距离

In [ ]:
# ====== 标签解析 ======

def parse_coreset(path):
    """解析 CoreSet.dat，返回 {pdbid: logKa} 字典。跳过以 '#' 开头的注释行。"""
    labels = {}
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            pdbid = parts[0]
            logKa = float(parts[3])
            labels[pdbid] = logKa
    print(f"从 CoreSet.dat 读取到 {len(labels)} 个复合物标签")
    return labels


# ---- 元素符号 -> one-hot 映射 (9类 + 1 芳香性 = 10维) ----
ELEMENT_LIST = ['C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br']  # 8 种常见元素 + 1 other


def atom_features(atom):
    """
    构建 10 维原子特征向量：
      - 前 9 维: 元素类型 one-hot (C, N, O, S, F, P, Cl, Br, other)
      - 第 10 维: 是否为芳香性原子
    """
    feat = np.zeros(ATOM_FEAT_DIM, dtype=np.float32)
    symbol = atom.GetSymbol()
    if symbol in ELEMENT_LIST:
        feat[ELEMENT_LIST.index(symbol)] = 1.0
    else:
        feat[8] = 1.0       # other 类别
    feat[9] = float(atom.GetIsAromatic())
    return feat


def load_mol(path, fmt):
    """
    用 RDKit 加载分子文件。
    fmt: 'mol2', 'sdf', 'pdb'
    先尝试正常加载，再尝试 sanitize=False 后手动 sanitize。
    """
    mol = None
    if fmt == 'mol2':
        mol = Chem.MolFromMol2File(path, sanitize=True)
        if mol is None:
            mol = Chem.MolFromMol2File(path, sanitize=False)
            if mol is not None:
                try:
                    Chem.SanitizeMol(mol)
                except Exception:
                    pass  # 保留 unsanitized 的分子
    elif fmt == 'sdf':
        supplier = Chem.SDMolSupplier(path, sanitize=True)
        for m in supplier:
            if m is not None:
                mol = m
                break
        if mol is None:
            supplier = Chem.SDMolSupplier(path, sanitize=False)
            for m in supplier:
                if m is not None:
                    mol = m
                    try:
                        Chem.SanitizeMol(mol)
                    except Exception:
                        pass
                    break
    elif fmt == 'pdb':
        mol = Chem.MolFromPDBFile(path, removeHs=False, sanitize=True)
        if mol is None:
            mol = Chem.MolFromPDBFile(path, removeHs=False, sanitize=False)
            if mol is not None:
                try:
                    Chem.SanitizeMol(mol)
                except Exception:
                    pass
    return mol

In [ ]:
# ====== 交互图构建 ======

def build_interaction_data(pdbid):
    """
    为单个蛋白-配体复合物构建交互图数据。

    返回:
      lig_feats  : (N_l, 10)  配体原子特征
      prot_feats : (N_p, 10)  蛋白口袋原子特征
      edge_index : (2, E)     交互边索引 [0]=配体原子, [1]=蛋白原子
      edge_dist  : (E, 1)     边距离
    如果解析失败则返回 None。
    """
    pocket_path = str(COMPLEX_DIR / pdbid / f"{pdbid}_pocket.pdb")
    ligand_mol2 = str(COMPLEX_DIR / pdbid / f"{pdbid}_ligand.mol2")
    ligand_sdf = str(COMPLEX_DIR / pdbid / f"{pdbid}_ligand.sdf")

    # ---- 1. 加载蛋白口袋 ----
    prot_mol = Chem.MolFromPDBFile(pocket_path, removeHs=True, sanitize=False)
    if prot_mol is None:
        return None
    try:
        Chem.SanitizeMol(prot_mol)
    except Exception:
        pass

    # ---- 2. 加载配体 (mol2 优先, sdf 备选) ----
    lig_mol = Chem.MolFromMol2File(ligand_mol2, sanitize=False)
    if lig_mol is None and os.path.exists(ligand_sdf):
        lig_mol = load_mol(ligand_sdf, 'sdf')
    if lig_mol is None:
        return None
    try:
        Chem.SanitizeMol(lig_mol)
    except Exception:
        pass

    # ---- 3. 去氢 ----
    try:
        prot_mol = Chem.RemoveHs(prot_mol)
    except Exception:
        pass
    try:
        lig_mol = Chem.RemoveHs(lig_mol)
    except Exception:
        pass

    # ---- 4. 提取 3D 坐标与原子特征 ----
    prot_conf = prot_mol.GetConformer()
    lig_conf = lig_mol.GetConformer()

    prot_coords = np.array([prot_conf.GetAtomPosition(i)
                            for i in range(prot_mol.GetNumAtoms())], dtype=np.float32)
    lig_coords = np.array([lig_conf.GetAtomPosition(i)
                           for i in range(lig_mol.GetNumAtoms())], dtype=np.float32)

    prot_feats = np.array([atom_features(a) for a in prot_mol.GetAtoms()], dtype=np.float32)
    lig_feats = np.array([atom_features(a) for a in lig_mol.GetAtoms()], dtype=np.float32)

    if prot_feats.shape[0] == 0 or lig_feats.shape[0] == 0:
        return None

    # ---- 5. 计算蛋白-配体距离矩阵，筛选交互边 ----
    # dist_mat[i, j] = 配体原子 i 到蛋白原子 j 的欧氏距离
    dist_mat = distance_matrix(lig_coords, prot_coords)
    lig_idx, prot_idx = np.where(dist_mat < DISTANCE_CUTOFF)

    if len(lig_idx) == 0:
        return None  # 没有交互边，无法构建交互图

    edge_index = np.stack([lig_idx, prot_idx], axis=0)      # (2, E)
    edge_dist = dist_mat[lig_idx, prot_idx].reshape(-1, 1)  # (E, 1)

    return lig_feats, prot_feats, edge_index, edge_dist


# ====== 构建全部交互图数据 ======
print("正在构建交互图数据...")
labels = parse_coreset(CORESET_FILE)

all_data = []
failed = 0
for pdbid, logKa in sorted(labels.items()):
    result = build_interaction_data(pdbid)
    if result is None:
        failed += 1
        continue
    lig_f, prot_f, ei, ed = result
    all_data.append((lig_f, prot_f, ei, ed, logKa))

print(f"成功加载 {len(all_data)} 个复合物, 失败 {failed} 个")

# ---- 展示第一个样本的数据概览 ----
sample = all_data[0]
sample_info = pd.DataFrame({
    '数据项': ['配体原子特征', '蛋白原子特征', '交互边索引', '边距离', '标签 (logKa)'],
    '形状': [str(sample[0].shape), str(sample[1].shape), str(sample[2].shape),
            str(sample[3].shape), str(sample[4])],
    '说明': [
        f'{sample[0].shape[0]} 个配体原子, 每个 {sample[0].shape[1]} 维特征',
        f'{sample[1].shape[0]} 个蛋白原子, 每个 {sample[1].shape[1]} 维特征',
        f'{sample[2].shape[1]} 条交互边 (配体原子 <-> 蛋白原子)',
        f'{sample[3].shape[0]} 条边的欧氏距离',
        '实验测量的结合亲和力'
    ]
})
print("\n第一个样本数据概览:")
display(sample_info)

## 3. 数据集与数据加载器

将预处理后的交互图数据封装为 PyTorch `Dataset`，并按照 80/20 比例随机划分训练集与测试集。

由于每个复合物的原子数和交互边数不同（变长图），`BATCH_SIZE` 设为 1 逐样本处理。

In [ ]:
class IGNDataset(Dataset):
    """交互图数据集。每个样本是一个蛋白-配体交互图 + 结合亲和力标签。"""
    def __init__(self, data_list):
        # data_list: list of (lig_feats, prot_feats, edge_index, edge_dist, label)
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        lig_f, prot_f, ei, ed, y = self.data[idx]
        return (torch.FloatTensor(lig_f),       # (N_l, 10)
                torch.FloatTensor(prot_f),       # (N_p, 10)
                torch.LongTensor(ei),            # (2, E)
                torch.FloatTensor(ed),           # (E, 1)
                torch.FloatTensor([y]))          # (1,)


# ---- 随机 80/20 划分训练/测试集 ----
indices = np.random.permutation(len(all_data))
split = int(0.8 * len(all_data))
train_idx, test_idx = indices[:split], indices[split:]

train_data = [all_data[i] for i in train_idx]
test_data = [all_data[i] for i in test_idx]

train_loader = DataLoader(IGNDataset(train_data), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(IGNDataset(test_data), batch_size=BATCH_SIZE, shuffle=False)

# ---- 展示数据集划分信息 ----
split_info = pd.DataFrame({
    '数据集': ['训练集', '测试集', '合计'],
    '样本数': [len(train_data), len(test_data), len(all_data)],
    '比例': [f'{len(train_data)/len(all_data):.0%}',
            f'{len(test_data)/len(all_data):.0%}',
            '100%']
})
display(split_info)

## 4. 模型架构

### IGNToyModel 逐层解析

IGN 的核心架构可以分为四个步骤：

```
输入: 配体原子特征 (N_l, 10) + 蛋白原子特征 (N_p, 10) + 交互边 (2, E) + 边距离 (E, 1)
  |
  v
Step 1: 共享节点嵌入 (Linear)
  |   配体原子 -> (N_l, hidden)
  |   蛋白原子 -> (N_p, hidden)
  v
Step 2: 边特征构建与边 MLP
  |   对每条交互边: [h_lig_i || h_prot_j || dist_ij] -> 边 MLP -> (E, hidden)
  v
Step 3: 边级别读出 (Edge-level Readout)
  |   注意力加权求和: attn(edge_h) * edge_h -> sum -> (1, hidden)
  |   最大池化: max(edge_h) -> (1, hidden)
  |   拼接: [weighted_sum || max_pool] -> (1, hidden*2)
  v
Step 4: 回归预测
  |   MLP -> 标量 logKa
  v
输出: 预测的 logKa
```

### 核心创新：边聚合而非节点聚合

传统 GNN（如 GCN、GAT）的消息传递在**节点**上进行：每个节点收集邻居信息，更新自身表示。而 IGN 直接在**交互边**上操作——因为蛋白-配体结合的关键信息就编码在这些接触界面上。注意力机制让模型自动学习哪些交互边（哪些原子对接触）对结合亲和力的贡献更大（例如氢键、疏水接触等）。

In [ ]:
class IGNToyModel(nn.Module):
    """
    交互图网络 (Interaction Graph Network) 玩具模型。

    核心思想：
      - 蛋白原子和配体原子共享同一个节点嵌入层；
      - 每条交互边 (配体原子 i <-> 蛋白原子 j) 的特征由
        [h_lig_i || h_prot_j || dist_ij] 拼接而成；
      - 边级别 MLP 处理后，通过注意力加权求和 + 最大池化得到
        全局图表示，最终回归预测 logKa。

    与传统 GNN 的区别：
      这里 **不做节点级消息传递**，而是直接在 **边上** 聚合信息，
      因为蛋白-配体的结合亲和力主要由接触界面（即边）决定。
    """
    def __init__(self, atom_dim=ATOM_FEAT_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        # 1. 共享节点嵌入：将原子特征映射到隐空间
        self.node_embed = nn.Linear(atom_dim, hidden_dim)

        # 2. 边 MLP：输入 = [h_lig_i || h_prot_j || dist_ij]
        #    拼接两个节点的隐表示与边距离 -> 边隐表示
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

        # 3. 边注意力：为每条边计算一个标量权重，用于加权求和
        self.attn_fc = nn.Linear(hidden_dim, 1)

        # 4. 预测头：[加权求和 || 最大池化] -> 标量 logKa
        self.predict = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, lig_feats, prot_feats, edge_index, edge_dist):
        """
        参数:
          lig_feats  : (N_l, atom_dim)  配体原子特征
          prot_feats : (N_p, atom_dim)  蛋白原子特征
          edge_index : (2, E)           交互边索引
          edge_dist  : (E, 1)           边距离

        返回:
          pred : 标量，预测的 logKa
        """
        # Step 1: 节点嵌入 -- 共享线性层
        lig_h = self.node_embed(lig_feats)      # (N_l, hidden)
        prot_h = self.node_embed(prot_feats)    # (N_p, hidden)

        # Step 2: 构建交互边特征
        #   从配体侧取出源节点嵌入，从蛋白侧取出目标节点嵌入
        src_feats = lig_h[edge_index[0]]        # (E, hidden) -- 配体原子
        dst_feats = prot_h[edge_index[1]]       # (E, hidden) -- 蛋白原子
        edge_input = torch.cat([src_feats, dst_feats, edge_dist], dim=-1)  # (E, hidden*2+1)
        edge_h = self.edge_mlp(edge_input)      # (E, hidden)

        # Step 3: 边级别读出 (edge-level readout)
        #   注意力加权求和 -- 让模型学习哪些交互边更重要
        attn = torch.tanh(self.attn_fc(edge_h))                    # (E, 1)
        weighted_sum = (attn * edge_h).sum(0, keepdim=True)        # (1, hidden)
        #   最大池化 -- 捕获最强的交互信号
        max_pool = edge_h.max(0, keepdim=True).values              # (1, hidden)
        graph_feat = torch.cat([weighted_sum, max_pool], dim=-1)   # (1, hidden*2)

        # Step 4: 回归预测 logKa
        return self.predict(graph_feat).squeeze()

In [ ]:
# ====== 实例化模型并展示参数量 ======

model = IGNToyModel(atom_dim=ATOM_FEAT_DIM, hidden_dim=HIDDEN_DIM).to(DEVICE)

# 统计各层参数量
param_rows = []
for name, param in model.named_parameters():
    param_rows.append({
        '层名称': name,
        '形状': str(list(param.shape)),
        '参数量': param.numel()
    })
param_rows.append({
    '层名称': '总计',
    '形状': '-',
    '参数量': sum(p.numel() for p in model.parameters())
})

param_df = pd.DataFrame(param_rows)
display(param_df)
print(f"\n模型总参数量: {sum(p.numel() for p in model.parameters()):,}")

## 5. 训练

### 损失函数

使用 **MSE Loss (均方误差)** 作为损失函数，衡量预测 logKa 与实验 logKa 之间的差距。

### 收敛监控

每 20 个 epoch 在测试集上计算一次验证损失，观察模型是否收敛。在 285 个样本的小数据集上，模型会很快过拟合，这里主要用于教学演示。

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

print(f"开始训练 {N_EPOCHS} 轮...\n")

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    train_losses = []

    for lig_f, prot_f, ei, ed, y in train_loader:
        # 每个 batch 只有 1 个样本 (变长图)，去掉 batch 维度
        lig_f = lig_f.squeeze(0).to(DEVICE)     # (N_l, 10)
        prot_f = prot_f.squeeze(0).to(DEVICE)   # (N_p, 10)
        ei = ei.squeeze(0).to(DEVICE)           # (2, E)
        ed = ed.squeeze(0).to(DEVICE)           # (E, 1)
        y = y.squeeze(0).to(DEVICE)             # (1,)

        # 跳过无交互边的样本（防御性检查）
        if ei.shape[1] == 0:
            continue

        optimizer.zero_grad()
        pred = model(lig_f, prot_f, ei, ed)
        loss = criterion(pred, y.squeeze())
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    # ---- 每 20 轮打印一次 ----
    if epoch % 20 == 0 or epoch == 1:
        model.eval()
        val_losses = []
        with torch.no_grad():
            for lig_f, prot_f, ei, ed, y in test_loader:
                lig_f = lig_f.squeeze(0).to(DEVICE)
                prot_f = prot_f.squeeze(0).to(DEVICE)
                ei = ei.squeeze(0).to(DEVICE)
                ed = ed.squeeze(0).to(DEVICE)
                y = y.squeeze(0).to(DEVICE)
                if ei.shape[1] == 0:
                    continue
                pred = model(lig_f, prot_f, ei, ed)
                val_losses.append(criterion(pred, y.squeeze()).item())

        avg_train = np.mean(train_losses) if train_losses else float('nan')
        avg_val = np.mean(val_losses) if val_losses else float('nan')
        print(f"  Epoch {epoch:>4d}/{N_EPOCHS}  |  Train Loss: {avg_train:.4f}  |  Val Loss: {avg_val:.4f}")

## 6. 测试与可视化

### 测试指标

| 指标 | 含义 | 范围 |
|------|------|------|
| **Pearson R** | 预测值与真实值的线性相关系数 | [-1, 1]，越接近 1 越好 |
| **RMSE** | 均方根误差，衡量预测误差的绝对大小 | >= 0，越小越好 |

### 可视化

散点图中每个点代表一个测试集复合物，横轴为实验 logKa，纵轴为预测 logKa。理想情况下所有点应落在 y = x 对角线上。

In [ ]:
# ====== 在测试集上收集预测结果 ======

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for lig_f, prot_f, ei, ed, y in test_loader:
        lig_f = lig_f.squeeze(0).to(DEVICE)
        prot_f = prot_f.squeeze(0).to(DEVICE)
        ei = ei.squeeze(0).to(DEVICE)
        ed = ed.squeeze(0).to(DEVICE)
        y = y.squeeze(0).to(DEVICE)
        if ei.shape[1] == 0:
            continue
        pred = model(lig_f, prot_f, ei, ed)
        y_true.append(y.item())
        y_pred.append(pred.item())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ---- 计算指标 ----
r, _ = pearsonr(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

# ---- 用 DataFrame 展示结果 ----
results_df = pd.DataFrame({
    '指标': ['测试集样本数', 'Pearson R', 'RMSE'],
    '值': [len(y_true), f'{r:.4f}', f'{rmse:.4f}']
})
print("测试集结果:")
display(results_df)

In [ ]:
# ====== 散点图：预测值 vs 真实值 ======

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_true, y_pred, alpha=0.6, edgecolors='k', linewidths=0.5, s=40)

# 画对角线 y = x
lo = min(y_true.min(), y_pred.min()) - 0.5
hi = max(y_true.max(), y_pred.max()) + 0.5
ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1.0, label='y = x')

ax.set_xlabel('Experimental logKa', fontsize=12)
ax.set_ylabel('Predicted logKa', fontsize=12)
ax.set_title(f'IGN Toy Model  |  R={r:.3f}, RMSE={rmse:.3f}', fontsize=13)
ax.legend(loc='upper left')
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 总结

本教程展示了 InteractionGraphNet (IGN) 的核心思想与实现流程，主要收获如下：

1. **交互图的构建**: 以空间距离阈值 (5.0 A) 筛选蛋白-配体原子对，将接触界面显式建模为图的边。这种表示方式直接捕获了结合位点的局部几何结构。

2. **边级别聚合 vs 节点级别聚合**: 与传统 GNN 不同，IGN 的核心操作在**边**上而非**节点**上。这是因为结合亲和力的物理本质来自蛋白-配体的原子间相互作用（即边），而非单个原子本身（即节点）。

3. **注意力机制的作用**: 通过对每条交互边学习注意力权重，模型能够自动识别哪些蛋白-配体原子对接触对结合亲和力的贡献更大（例如氢键、疏水接触等）。

4. **局限性**: 本教程为简化版玩具模型，使用了简化的原子特征和较小的数据集（285 个样本）。完整版 IGN 还包括更丰富的原子/键特征、多层消息传递、以及更大规模的训练数据。

**进阶阅读**: 如需了解完整版 IGN 的实现细节，请参考 `source_repos/InteractionGraphNet/` 中的原始代码。